# **3. Baseline Model**

In [69]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sidetable
import sklearn
import feature_engine
import scipy
from scipy import stats
from pathlib import Path
import pickle

%matplotlib inline
sns.set_style('darkgrid')


In [70]:
# Display Settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [71]:
# define file path for processed data
parent_path = Path.cwd().parent
file_path = parent_path.joinpath("models", "processed.pkl")

In [72]:
# Load the processed data from the previous step
print(f"Loading processed data...")
with open(file_path, "rb") as f:
    processed_data = pickle.load(f)

Loading processed data...


In [73]:
processed_data.keys()

dict_keys(['X_train', 'y_train', 'X_test', 'test_id', 'numeric_features', 'ordinal_features', 'categorical_features', 'boolean_features', 'year_features'])

In [74]:
# Extract features and target variable
X_train = processed_data["X_train"]
y_train = processed_data["y_train"]
X_test = processed_data["X_test"]
numeric_feat = processed_data["numeric_features"]
ordinal_feat = processed_data["ordinal_features"]
categorical_feat = processed_data["categorical_features"]
boolean_feat = processed_data["boolean_features"]
year_feat = processed_data["year_features"]

In [75]:
print(f"Training data shape: {X_train.shape}")
print(f"Test data shape: {X_test.shape}")

Training data shape: (1459, 23)
Test data shape: (1457, 23)


#### **1. Feature Selection**

In [76]:
# 1. Feature Selection for Linear Models
print("\n=== FEATURE SELECTION ===")

# Create dictionary to store different feature sets
feature_sets = {}


=== FEATURE SELECTION ===


**Preference 1.1. Correlation Analysis: compute correlation between numeric features and the target variable**


In [ ]:
correlations = X_train[ordinal_feat + numeric_feat + year_feat + boolean_feat].corrwith(y_train).abs().sort_values(ascending=False)
print("Top correlated features with the target variable:")
print(correlations)

Top correlated features with the target variable:
LivAreaQual                0.795007
OverallQual                0.793055
TotalSF                    0.765099
NeighborhoodMedianPrice    0.720926
ExterQual                  0.684381
KitchenQual                0.660666
GarageCars                 0.640954
TotalBath                  0.633484
BsmtQual                   0.585748
HouseAge                   0.573607
QualCond                   0.565831
GarageFinish               0.549590
RemodeledAge               0.513813
GarageYrBlt                0.508240
HasFireplace               0.472022
Fireplaces                 0.466967
HasPorch                   0.413045
MasVnrArea                 0.405516
TotalPorchSF               0.337434
CentralAir                 0.251326
HasGarage                  0.236829
BsmtFinSF1                 0.185520
dtype: float64


**Prefrence 1.2. Use Ridge Regression to compute feature importance**


In [78]:
from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge

# Transformes to cast boolean to int
bool_to_int = FunctionTransformer(lambda x: x.astype(int))

# Define preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_feat),
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_feat),
        ("boolean", bool_to_int, bool_feat),
        ("ordinal", "passthrough", ordinal_feat),
        ("year", "passthrough", year_feat)
    ]
)

# build pipeline
ridge_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("ridge", Ridge(alpha=10.0))
])
ridge_pipeline.fit(X_train, y_train)

# Extract features names after one-hot encoding
features_name = (
    numeric_feat +
    list(ridge_pipeline.named_steps['preprocessor'].named_transformers_['categorical'].get_feature_names_out(categorical_feat)) +
    bool_feat + ordinal_feat + year_feat
) 

# Extract coefficients from the Ridge model
ridge_coef = ridge_pipeline.named_steps['ridge'].coef_

feature_importance = pd.DataFrame({
    "Feature": features_name,
    "Importance": ridge_coef
})
feature_importance.sort_values(by="Importance", ascending=False, inplace=True, ignore_index=True)
print("Top 20 features by importance from Ridge Regression:")
display(feature_importance.head(20))


Top 20 features by importance from Ridge Regression:


,Feature,Importance
0,Neighborhood_NoRidge,27612.306051
1,TotalSF,20340.434725
2,Neighborhood_StoneBr,18199.480640
3,GarageCars,15816.627446
4,Fireplaces,14545.714476
5,Neighborhood_BrkSide,13638.128931
6,NeighborhoodMedianPrice,13187.580203
7,Neighborhood_MeadowV,11110.197771
8,Neighborhood_NridgHt,10705.531313
9,Neighborhood_IDOTRR,10599.184349


In [ ]:
# Feature Set 2: Top features based on Ridge importance
# Get top 20 features from ridge importance
top_ridge_features = feature_importance['Feature'][:20].tolist()

# Extract original name of features
# E.g., from "Neighborhood_Sawyer" to "Neighborhood"
original_cat_feat = set()
for f in feature_importance['Feature']:
    for cat in categorical_feat:
        if f.startswith(cat + '_'):
            original_cat_feat.add(cat)
            break

# Filter top ridge features to get numeric features, 
top_ridge_numeric = [f for f in numeric_feat if f in top_ridge_features]
top_ridge_boolean = [f for f in boolean_feat if f in top_ridge_features]
top_ridge_year = [f for f in year_feat if f in top_ridge_features]
top_ridge_ordinal = [f for f in ordinal_feat if f in top_ridge_features]

# Find top categorical features by max importance of their encoded versions
if categorical_feat:
    cat_max_importance = (
        feature_importance
        .assign(Original=feature_importance['Feature'].str.split('_').str[0])
        .groupby('Original')['Importance']
        .max()
        .loc[categorical_feat]
        .sort_values(ascending=False)
    )
    top_ridge_categorical_original = cat_max_importance.index[:5].tolist()

# Store the second feature set - handle case with no categorical features
ridge_numeric_feat = top_ridge_numeric.copy() + top_ridge_boolean.copy() + top_ridge_year.copy() + top_ridge_ordinal.copy()
if top_ridge_categorical_original:
    ridge_numeric_feat.extend(top_ridge_categorical_original)

feature_sets['ridge_selected'] = {
    'numeric_features': top_ridge_numeric,
    'boolean_features' : top_ridge_boolean,
    'ordinal_features' : top_ridge_ordinal,
    'year_features' : top_ridge_year,
    'categorical_features': top_ridge_categorical_original,
    'X_train': X_train[ridge_numeric_feat],
    'X_test': X_test[ridge_numeric_feat]
}

**Preference 1.3. Selection of Top Features Manually**


In [83]:
selected_features = [
    'LivAreaQual',              # Living area quality
    'OverallQual',              # Overall quality
    'TotalSF',                  # Total square feet
    'NeighborhoodMedianPrice',   # Neighborhood price level
    'ExterQual',                # Exterior quality
    'KitchenQual',              # Kitchen quality
    'BsmtQual',                 # Basement quality
    'GarageCars',                # Garage capacity
    'TotalBath',                # Total bathrooms
    'HouseAge',                 # House age
    'QualCond',                 # Quality and condition
    'RemodeledAge',             # Years since remodeling
    'HasFireplace',             # Has fireplace indicator
    'CentralAir',               # Central air indicator
    'GarageFinish'             # Finishing of garage
]

# Check if all selected features exist in the dataset
missing_features = [feat for feat in selected_features if feat not in X_train.columns]
if missing_features:
    print(f"Warning: These selected features are missing from the dataset: {missing_features}")
    # Remove missing features from selection
    selected_features = [feat for feat in selected_features if feat in X_train.columns]

# Categorize manual features
manual_numeric = [f for f in selected_features if f in numeric_feat]
manual_ordinal = [f for f in selected_features if f in ordinal_feat]
manual_year = [f for f in selected_features if f in year_feat]
manual_boolean = [f for f in selected_features if f in boolean_feat]
manual_categorical = [f for f in selected_features if f in categorical_feat]

# Store the third feature set
feature_sets['manual_selected'] = {
    'numeric_features': manual_numeric,
    'boolean_features': manual_boolean,
    'ordinal_features': manual_ordinal,
    'year_features': manual_year,
    'categorical_features': manual_categorical,
    'X_train': X_train[selected_features],
    'X_test': X_test[selected_features]
}

# Print summary of feature sets
for name, feature_set in feature_sets.items():
    print(f"\nFeature set: {name}")
    print(f"Number of features: {len(feature_set['numeric_features']) + len(feature_set['categorical_features']) + 
                                len(feature_set['boolean_features']) + len(feature_set['ordinal_features']) + 
                                len(feature_set['year_features'])}"
                                )
    print(f"Numeric features: {len(feature_set['numeric_features'])}")
    print(f"Boolean features: {len(feature_set['boolean_features'])}")
    print(f"Ordianl features: {len(feature_set['ordinal_features'])}")
    print(f"Year features: {len(feature_set['year_features'])}")
    print(f"Categorical features: {len(feature_set['categorical_features'])}")


Feature set: ridge_selected
Number of features: 13
Numeric features: 5
Boolean features: 0
Ordianl features: 7
Year features: 0
Categorical features: 1

Feature set: manual_selected
Number of features: 15
Numeric features: 6
Boolean features: 2
Ordianl features: 7
Year features: 0
Categorical features: 0


## **2. Processing Pipeline**

In [84]:
preprocessing_pipelines = {}